3) HuggingFaceEmbeddings

HuggingFaceEndpointEmbeddings

In [ ]:
# [목적] Hugging Face 임베딩 예제에 필요한 API 키 환경을 불러옵니다.
# load_dotenv는 .env 파일의 HF_TOKEN 같은 비밀 값을 현재 실행 환경에 등록합니다.
# API 키를 코드에 직접 쓰지 않고, 이후 Hugging Face Endpoint에 안전하게 연결하기 위해 필요합니다.
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# [목적] 임베딩 예제의 추적·경고·모델 캐시 환경을 설정합니다.
# LangSmith 기록을 켜고 불필요한 경고를 숨긴 뒤, Hugging Face 모델 파일을 ./cache/에 저장하도록 지정합니다.
# 실행 내역을 확인하고 다운로드한 모델을 재사용해 예제를 편리하게 실행하기 위한 준비 단계입니다.
from langchain_teddynote import logging
import os
import warnings

logging.langsmith("Chapter11-Embeddings")

warnings.filterwarnings("ignore")

os.environ["HF_HOME"] = "./cache/"

In [ ]:
# [목적] 여러 임베딩 모델의 의미 검색 성능을 비교할 예제 문장을 준비합니다.
# 한국어·영어 문장을 texts 목록에 모아 문서 임베딩 입력으로 전달합니다.
# 뒤에서 질문 벡터와 비교하여 의미가 가까운 문서를 찾기 위한 기준 데이터입니다.
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

In [ ]:
# [목적] Hugging Face 서버에서 임베딩을 생성하는 Endpoint 방식의 모델을 설정합니다.
# HuggingFaceEndpointEmbeddings는 로컬에 모델을 받지 않고 API로 feature-extraction 작업을 요청합니다.
# HF_TOKEN으로 인증한 뒤, 다국어 문장 의미를 벡터로 바꾸기 위해 사용합니다.
import os
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name,
    task="feature-extraction",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
)

In [ ]:
# [목적] 예제 문서들을 Hugging Face Endpoint 임베딩 벡터로 변환합니다.
# embed_documents가 texts의 각 문장을 숫자 벡터로 만들고, %%time이 셀 전체 실행 시간을 측정합니다.
# 생성된 벡터는 질문과 문서의 의미적 유사도를 계산하는 데 사용합니다.
%%time
embedded_documents = hf_embeddings.embed_documents(texts)

In [ ]:
# [목적] Endpoint 임베딩에 사용한 모델과 벡터 차원 수를 확인합니다.
# embedded_documents[0]은 첫 문서의 벡터이고, len으로 그 안의 숫자 개수를 셉니다.
# 검색에 사용할 벡터의 크기가 예상한 모델 설정과 맞는지 점검합니다.
print("[HuggingFace Endpoint Embedding]")
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

In [ ]:
# [목적] 사용자 질문을 검색어용 임베딩으로 변환해 결과를 확인합니다.
# embed_query는 질문 한 건을 문서 벡터와 비교 가능한 숫자 벡터로 바꿉니다.
# 이 벡터는 다음 셀에서 각 문서와의 유사도를 계산하는 기준이 됩니다.
embedded_query = hf_embeddings.embed_query("LangChain에 대해서 알려주세요.")
embedded_query

임베딩된 질문과 문서 간의 유사도 계산하기

In [ ]:
# [목적] 질문과 모든 문서 벡터의 내적 점수를 한 번에 계산합니다.
# NumPy 배열로 바꾼 뒤 @ 연산자와 전치(.T)를 사용해 질문 벡터와 각 문서 벡터를 비교합니다.
# 정규화된 벡터에서는 이 점수가 코사인 유사도처럼 의미적 가까움을 나타냅니다.
import numpy as np

np.array(embedded_query) @ np.array(embedded_documents).T

In [ ]:
# [목적] 질문과 가장 유사한 문서부터 순서를 정하는 인덱스를 만듭니다.
# argsort는 점수를 작은 순서의 위치 번호로 만들고, [::-1]은 이를 역순으로 뒤집습니다.
# 정렬된 위치 번호는 다음 셀에서 검색 결과 문장을 높은 점수 순으로 보여주는 데 사용합니다.
sorted_idx = (
    np.array(embedded_query) @ np.array(embedded_documents).T
).argsort()[::-1]

sorted_idx

In [ ]:
# [목적] 유사도 순서에 따라 질문과 관련된 문서를 출력합니다.
# enumerate로 순위와 문서 위치를 함께 가져오고, texts[idx]로 해당 원문을 선택합니다.
# 임베딩 검색이 의미상 가까운 문장을 위에 배치하는지 눈으로 확인할 수 있습니다.
print("[Query] LangChain 에 대해서 알려주세요.\n========================================")
for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

HuggingFaceEmbeddings

In [ ]:
# [목적] 로컬 컴퓨터에서 실행하는 Hugging Face 임베딩 모델을 설정합니다.
# HuggingFaceEmbeddings가 모델을 내려받아 CPU에서 실행하고, normalize_embeddings가 벡터 길이를 맞춥니다.
# 외부 Endpoint 대신 로컬 환경에서 문서 임베딩을 만들고 싶을 때 사용하는 방식입니다.
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cpu"},  # cuda, cpu
    encode_kwargs={"normalize_embeddings": True},
)

In [ ]:
# [목적] 로컬 Hugging Face 모델로 예제 문서를 임베딩합니다.
# embed_documents가 texts 목록 전체를 벡터 목록으로 변환하며, %%time이 소요 시간을 재는 셀 매직입니다.
# Endpoint 방식과 로컬 방식의 실행 결과와 속도를 비교하는 데 사용합니다.
%%time

embedded_documents = hf_embeddings.embed_documents(texts)

In [ ]:
# [목적] 로컬 모델의 이름과 생성된 임베딩 벡터 크기를 확인합니다.
# 첫 문서 벡터의 숫자 개수를 출력해 모델이 만든 차원 수를 확인합니다.
# 이후 다른 모델의 임베딩 차원과 비교하거나 벡터 저장 공간을 판단하는 데 도움이 됩니다.
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

BGE-M3 임베딩

In [ ]:
# [목적] BAAI/bge-m3를 LangChain 방식으로 실행해 임베딩을 생성합니다.
# 모델·실행 장치·정규화 옵션을 지정한 뒤 문서를 벡터로 바꾸고 모델 정보와 차원을 출력합니다.
# 다국어 검색에 널리 쓰이는 BGE-M3를 앞선 E5 모델과 같은 흐름으로 활용하는 예제입니다.
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-m3"

model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

%time
embedded_documents = hf_embeddings.embed_documents(texts)

print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

In [ ]:
# [목적] BGE-M3 임베딩으로 질문과 문서의 유사도 순위를 계산해 출력합니다.
# 질문·문서를 벡터로 바꾼 뒤 내적 점수를 정렬하고, 높은 점수의 원문부터 표시합니다.
# 모델을 바꿔도 같은 벡터 검색 흐름을 적용할 수 있음을 보여줍니다.
import numpy as np

embedded_query = hf_embeddings.embed_query("LangChain에 대해서 알려주세요.")
embedded_documents = hf_embeddings.embed_documents(texts)

# 질문(embedded_query): LangChain에 대해서 알려주세요.
np.array(embedded_query) @ np.array(embedded_documents).T

sorted_idx = (
    np.array(embedded_query) @ np.array(embedded_documents).T
).argsort()[::-1]

print("[Query] LangChain에 대해서 알려주세요. \n========================================")

for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

FlagEmbedding

In [ ]:
# [목적] FlagEmbedding 라이브러리로 BGE-M3의 밀집 벡터를 직접 생성합니다.
# BGEM3FlagModel의 encode가 문장들을 처리하고, dense_vecs에서 문장 전체를 대표하는 벡터만 꺼냅니다.
# LangChain 래퍼 없이 BGE-M3의 여러 임베딩 기능을 직접 사용하는 출발점입니다.
from FlagEmbedding import BGEM3FlagModel

model_name = "BAAI/bge-m3"

bge_embeddings = BGEM3FlagModel(
    model_name,
    use_fp16=False
)

bge_embedded = bge_embeddings.encode(
    texts,
    batch_size=12,
    max_length=8192,
)["dense_vecs"]

In [ ]:
# [목적] 직접 생성한 BGE-M3 밀집 벡터 배열의 형태를 확인합니다.
# shape는 행 수(문장 수)와 열 수(각 벡터의 차원)를 튜플로 반환합니다.
# 모든 입력 문장이 같은 크기의 벡터로 변환되었는지 점검하기 위해 사용합니다.
bge_embedded.shape

In [ ]:
# [목적] BGE-M3에서 밀집 벡터만 반환하도록 설정해 인코딩합니다.
# return_dense=True는 문장 전체 의미를 나타내는 dense_vecs 결과를 요청하는 옵션입니다.
# 이후 결과 딕셔너리에서 밀집 벡터의 차원을 확인하기 위해 사용합니다.
from FlagEmbedding import BGEM3FlagModel

bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=False
)

bge_encoded = bge_flagmodel.encode(texts, return_dense=True)

In [ ]:
# [목적] 반환된 BGE-M3 밀집 벡터의 배열 크기를 확인합니다.
# bge_encoded는 여러 결과를 담은 딕셔너리이며, dense_vecs 키가 문장 대표 벡터를 가리킵니다.
# BGE-M3가 생성한 벡터 수와 차원 수를 간단히 점검합니다.
bge_encoded["dense_vecs"].shape

In [ ]:
# [목적] BGE-M3에서 단어 단위 가중치인 희소 벡터를 생성합니다.
# return_sparse=True는 문장에 등장한 중요한 단어와 가중치를 lexical_weights로 반환하도록 요청합니다.
# 의미 중심의 밀집 벡터와 달리, 특정 단어의 일치 정도를 검색에 반영할 때 사용합니다.
bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=False
)

bge_encoded = bge_flagmodel.encode(
    texts,
    return_sparse=True
)

In [ ]:
# [목적] 희소 벡터를 이용해 단어 수준의 일치 점수를 비교합니다.
# compute_lexical_matching_score가 첫 문서와 자기 자신, 첫 문서와 두 번째 문서의 단어 가중치를 비교합니다.
# 같은 문서와 다른 문서의 점수 차이로 희소 검색 점수의 의미를 확인합니다.
lexical_scores1 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded["lexical_weights"][0],
    bge_encoded["lexical_weights"][0]
)

lexical_scores2 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded["lexical_weights"][0],
    bge_encoded["lexical_weights"][1]
)

print(lexical_scores1)  # 0 <-> 0
print(lexical_scores2)  # 0 <-> 1

In [ ]:
# [목적] BGE-M3에서 ColBERT 방식의 토큰별 벡터를 생성합니다.
# return_colbert_vecs=True는 문장 전체가 아닌 각 토큰을 표현하는 벡터 묶음을 반환합니다.
# 세밀한 토큰 단위 비교가 필요한 재순위화나 고급 검색에 활용할 수 있습니다.
bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=False
)

bge_encoded = bge_flagmodel.encode(
    texts,
    return_colbert_vecs=True
)

In [ ]:
# [목적] ColBERT 토큰 벡터로 문장 쌍의 세밀한 유사도 점수를 비교합니다.
# colbert_score가 첫 문서와 자기 자신, 그리고 첫 문서와 두 번째 문서의 토큰 벡터를 각각 비교합니다.
# 밀집·희소 벡터와 다른 ColBERT 점수 방식의 검색 활용을 확인하는 예제입니다.
colbert_scores1 = bge_flagmodel.colbert_score(
    bge_encoded["colbert_vecs"][0],
    bge_encoded["colbert_vecs"][0]
)

colbert_scores2 = bge_flagmodel.colbert_score(
    bge_encoded["colbert_vecs"][0],
    bge_encoded["colbert_vecs"][1]
)

print(colbert_scores1)  # 0 <-> 0
print(colbert_scores2)  # 0 <-> 1p